**Github** : https://github.com/OzgurYldrm/AI-ML-Course     
**Youtube** : https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence

https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

In [1]:
import pandas as pd
import unicodedata
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

pd.set_option("display.max_colwidth", None)

In [2]:
data = pd.read_csv("spam.csv",encoding="latin-1")

# EDA

In [4]:
data.head(5)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN


In [5]:
data = data.drop(["Unnamed: 2","Unnamed: 3","Unnamed: 4"],axis=1)
data["v1"] = data["v1"].map({
    "ham": 0,
    "spam": 1
})

In [6]:
data.head(5)

,v1,v2
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"


In [11]:
print(data[data["v1"]==1].sample(1)["v2"])

4204    IMPORTANT INFORMATION 4 ORANGE USER 0796XXXXXX. TODAY IS UR LUCKY DAY!2 FIND OUT WHY LOG ONTO http://www.urawinner.com THERE'S A FANTASTIC PRIZEAWAITING YOU!
Name: v2, dtype: object


In [12]:
pos_samples = data[data["v1"] == 1].sample(5, random_state=42)
neg_samples = data[data["v1"] == 0].sample(5, random_state=42)

samples = pd.concat([pos_samples, neg_samples])

samples[["v1", "v2"]]

,v1,v2
1455,1,Summers finally here! Fancy a chat or flirt with sexy singles in yr area? To get MATCHED up just reply SUMMER now. Free 2 Join. OptOut txt STOP Help08714742804
1852,1,"This is the 2nd time we have tried 2 contact u. U have won the 750 Pound prize. 2 claim is easy, call 08718726970 NOW! Only 10p per min. BT-national-rate"
672,1,Get ur 1st RINGTONE FREE NOW! Reply to this msg with TONE. Gr8 TOP 20 tones to your phone every week just å£1.50 per wk 2 opt out send STOP 08452810071 16
946,1,Ur cash-balance is currently 500 pounds - to maximize ur cash-in now send GO to 86688 only 150p/msg. CC: 08718720201 PO BOX 114/14 TCR/W1
2879,1,"Last Chance! Claim ur å£150 worth of discount vouchers today! Text SHOP to 85023 now! SavaMob, offers mobile! T Cs SavaMob POBOX84, M263UZ. å£3.00 Sub. 16"
3714,0,"I am late,so call you tomorrow morning.take care sweet dreams....u and me...ummifying...bye."
1311,0,U r too much close to my heart. If u go away i will be shattered. Plz stay with me.
548,0,Wait &lt;#&gt; min..
1324,0,Can you call me plz. Your number shows out of coveragd area. I have urgnt call in vasai &amp; have to reach before 4'o clock so call me plz
3184,0,MAYBE IF YOU WOKE UP BEFORE FUCKING 3 THIS WOULDN'T BE A PROBLEM.


# Preprocess

In [14]:
data["v2"] = (data["v2"].astype(str)
    .str.lower() #Lowercasing
    .map(lambda x: unicodedata.normalize("NFKC", x)) #Unicode normalization
    .str.replace(r'http\S+|www\S+', ' <URL> ', regex=True)
    .str.replace(r'\S+@\S+', ' <EMAIL> ', regex=True)
    .str.replace(r'\+?\d[\d\s\-]{7,}', ' <PHONE> ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(r'<.*?>', ' ', regex=True)  # html tag
    .str.strip()
)

In [15]:
data

,v1,v2
0,0,"go until jurong point, crazy.. available only in bugis n great world la e buffet... cine there got amore wat..."
1,0,ok lar... joking wif u oni...
2,1,free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005. text fa to 87121 to receive entry question(std txt rate)t&c's apply over18's
3,0,u dun say so early hor... u c already then say...
4,0,"nah i don't think he goes to usf, he lives around here though"
...,...,...
5567,1,"this is the 2nd time we have tried 2 contact u. u have won the å£750 pound prize. 2 claim is easy, call now1! only 10p per minute. bt-national-rate."
5568,0,will ì_ b going to esplanade fr home?
5569,0,"pity, * was in mood for that. so...any other suggestions?"
5570,0,the guy did some bitching but i acted like i'd be interested in buying something else next week and he gave it to us for free


In [16]:
unique_words = set(
    word
    for text in data["v2"]
    for word in text.split()
)

print(len(unique_words))

avg_len = data["v2"].str.split().str.len().mean()
print(avg_len)

13102
15.395369705671213


# Model

In [17]:
from sklearn.model_selection import train_test_split

X = data["v2"]
y = data["v1"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=48151623)

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9)
results = tfidf.fit_transform(X_train)
results[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14 stored elements and shape (1, 11806)>

In [19]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2), #Bigram da tutyoruz
        min_df=2, #en az kaç dökümanda geçmesi gerekiyor: "asdasdasd" elenir
        max_df=0.9 #en fazla kaç dökümanda geçmesi gerekiyor: "the" elenir
    )),
    ("svm", LinearSVC(class_weight="balanced"))
])

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=["accuracy","f1"],
    n_jobs=-1
)

print("CV Mean ACC:", scores["test_accuracy"].mean())
print("CV Mean F1 :", scores["test_f1"].mean())

CV Mean ACC: 0.9872126769788885
CV Mean F1 : 0.9515036886791087


In [21]:
pipe.fit(X_train, y_train) #Nihai model

,steps,"[('tfidf', ...), ('svm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [22]:
preds = pipe.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.94      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [24]:
preds = pipe.predict(X_test)

result_df = pd.DataFrame({
    "text": X_test.values,
    "true": y_test.values,
    "pred": preds
})

result_df.tail(10)

,text,true,pred
1105,"shop till u drop, is it you, either 10k, 5k, å£500 cash or å£100 travel voucher, call now, . ntt po box cr01327bt fixedline cost 150ppm mobile vary",1,1
1106,oh k.i think most of wi and nz players unsold.,0,0
1107,sorry. you never hear unless you book it. one was kinda a joke--thet were really looking for skinny white girls. the other was one line--you can only do so much on camera with that. something like that they're casting on the look.,0,0
1108,hmm thinking lor...,0,0
1109,"i.ll always be there, even if its just in spirit. i.ll get a bb soon. just trying to be sure i need it.",0,0
1110,are there ta jobs available? let me know please cos i really need to start working,0,0
1111,"mmmmmm ... i love you,so much, ahmad ... i can't wait for this year to begin as every second takes me closer to being at your side. happy new year, my love!!",0,0
1112,if he started searching he will get job in few days.he have great potential and talent.,0,0
1113,"hi there, 2nights ur lucky night! uve been invited 2 xchat, the uks wildest chat! txt chat to 86688 now! 150p/msgrcvdhg/suite342/2lands/row/w1j6hl ldn 18yrs",1,1
1114,bring tat cd don forget,0,0
